# ECON 662 — Problem Set 2
## Part (b): Hotz-Miller (1993) Inversion — MLE

Estimate $(\theta, \rho, \sigma_\rho)$ without solving the value function in the estimation algorithm, but relying on the Hotz-Miller (1993) inversion. Estimate the parameters with MLE.

### The core idea 

In Part (a) NFXP, for each guess of $\theta$ the inner loop solves a fixed point:
$$V = \ln\!\Bigl(\sum_j \exp(\delta_j)\Bigr) + \gamma$$
where $V$ appears on both sides, requiring iterative contraction until convergence.

Under Type I EV errors, the conditional expectation of the shock given action $j$ is selected satisfies:
$$\phi_j(s) \equiv \mathbb{E}[\varepsilon_j \mid j \text{ selected}] = \gamma - \ln Pr_j^*(s)$$
This lets us write the ex-ante value function as a CCP-weighted average:
$$V(s) = \sum_{j \in J} Pr_j^*(s) \cdot \bigl(\delta_j(s) + \phi_j(s)\bigr)$$
where $\delta_j(s) = \bar{u}(s,j;\theta) + \beta\,EV(s,j)$ is the choice-specific value.

Expanding $EV(s,j) = \sum_{s'} TP(j,s)_{s,s'} V(s')$ and collecting $V$ on the left gives the Hotz-Miller linear system:

$$\boxed{V(\mathbf{s}) = \Bigl[I_{|S|\times|S|} - \beta \sum_{j} P_j(\mathbf{s}) \cdot TP(j,\mathbf{s})\Bigr]^{-1} \times \Bigl[\sum_{j} P_j(\mathbf{s}) \cdot \bigl(u_j(\mathbf{s}) + \phi_j(\mathbf{s})\bigr)\Bigr]}$$

**Notation:**
- $P_j(\mathbf{s})$: $|S|$-vector of $\hat{P}(a=j|s)$ : observed from data, fixed before optimizer
- $TP(j, \mathbf{s})$: $|S|\times|S|$ transition matrix conditional on action $j$ : built from data, fixed
- $u_j(\mathbf{s})$: $|S|$-vector of deterministic payoffs : depends on $\theta$
- $\phi_j(\mathbf{s}) = \gamma - \ln P_j(\mathbf{s})$: logit surplus : fixed from data
- $P_j(\mathbf{s}) \cdot TP(j,\mathbf{s})$: row-wise scaling (row $s$ of $TP$ scaled by scalar $P_j(s)$)

**Four-step procedure**
1. Estimate transition matrix : exogenous (RC, empirical counts) + endogenous (mileage)
2. Estimate CCP $\hat{P}(a=j|s)$ from data (frequency estimator)
3. For each guess of $\theta$: $V \leftarrow (\widehat{CCP},\,\theta)$ via HM inversion; get $P$ from $V$
4. Calculate likelihood given current $\theta$

**RC discretization: empirical bins**

Same as Part (a): we cut the observed RC range into $N_{RC}$ equal-width bins and represent each
by its midpoint. The transition matrix $\Pi$ is built by counting empirical transitions in the data.
This is fixed once and never recomputed during optimization, no parametric normal CDF, no re-evaluation at each optimizer call.

#### Import Libraries

In [1]:
import time
import numpy as np
import pandas as pd
from scipy.optimize import minimize

np.random.seed(42)

#### Load Data

In [2]:
# Variables: i (bus id), t (month 1-100), a (0=maintain, 1=replace),
#            x (mileage state 1-7), rc (replacement cost, continuous)
df = pd.read_csv("ddc_pset.csv")
df = df.sort_values(["i", "t"]).reset_index(drop=True)

print(f"Loaded {len(df):,} observations for {df['i'].nunique():,} buses.")
print(f"RC range: [{df['rc'].min():.4f}, {df['rc'].max():.4f}]")
print(f"Replacement rate: {df['a'].mean():.4f}")
df.head(10)

Loaded 100,000 observations for 1,000 buses.
RC range: [31.5000, 62.5000]
Replacement rate: 0.1724


,i,t,a,x,rc
0,1,1,0,4,45.0
1,1,2,0,5,49.0
2,1,3,0,6,54.0
3,1,4,1,7,47.0
4,1,5,0,1,56.0
5,1,6,0,2,52.5
6,1,7,0,3,55.0
7,1,8,0,4,50.5
8,1,9,0,5,52.0
9,1,10,0,6,42.0


#### Global Constants

In [3]:
BETA        = 0.95          # discount factor (given)
N_X         = 7             # mileage states {1,...,7}
N_RC_BINS   = 8             # RC bins (same as Part a)
S           = N_X * N_RC_BINS  # |S| = 56 total states
GAMMA       = 0.5772156649  # Euler-Mascheroni constant

# State ordering: s = xi * N_RC_BINS + j,  xi in {0,...,6},  j in {0,...,7}
print(f"|S| = {N_X} x {N_RC_BINS} = {S} states")

|S| = 7 x 8 = 56 states


### Estimate Transition Matrix 

Same approach as Part (a). We:

1. Cut the observed RC range into $N_{RC}=8$ equal-width bins; each bin represented by its midpoint.
2. Count empirical transitions $(bin_{t-1} \to bin_t)$ within each bus to get count matrix $C$.
3. Normalize rows: $\Pi_{j,j'} = C_{j,j'} / \sum_{j''} C_{j,j''}$.

$\Pi$ is built once from the data and never touched again during optimization.

We also estimate $(\rho_0, \rho_1, \sigma_\rho)$ via OLS — the AR(1) likelihood is separable
from the choice likelihood, so these are fixed before the optimizer runs.

In [4]:
# Equal-width bins over [RC_min, RC_max]  (same as Part a)
rc_min      = df['rc'].min()
rc_max      = df['rc'].max()
bin_edges   = np.linspace(rc_min, rc_max, N_RC_BINS + 1)        # shape (9,)
bin_midpoints = (bin_edges[:-1] + bin_edges[1:]) / 2.0           # shape (8,)
rc_grid     = bin_midpoints   # alias: representative RC value per bin

# Assign each observation to its bin (0-indexed)
rc_bin_idx  = np.searchsorted(bin_edges[1:], df['rc'].values, side='left')
rc_bin_idx  = np.clip(rc_bin_idx, 0, N_RC_BINS - 1)
df['rc_bin'] = rc_bin_idx
df['rc_mid'] = bin_midpoints[rc_bin_idx]

print(f"Bin edges: {np.round(bin_edges, 3)}")
print(f"Bin midpoints (rc_grid): {np.round(rc_grid, 3)}")

Bin edges: [31.5   35.375 39.25  43.125 47.    50.875 54.75  58.625 62.5  ]
Bin midpoints (rc_grid): [33.438 37.312 41.188 45.062 48.938 52.812 56.688 60.562]


In [5]:
# Build empirical transition count matrix C (N_RC_BINS x N_RC_BINS)
# C[j, j'] = # times RC moved from bin j to bin j' in consecutive periods
C = np.zeros((N_RC_BINS, N_RC_BINS))
for _, g in df.groupby("i"):
    bins = g['rc_bin'].values
    for j_from, j_to in zip(bins[:-1], bins[1:]):
        C[j_from, j_to] += 1

# Normalize rows -> empirical transition probability matrix Pi
Pi = C / C.sum(axis=1, keepdims=True)   # shape (8, 8)

print(f"Pi ({N_RC_BINS}x{N_RC_BINS}) — built from data, FIXED for entire estimation:")
print(np.round(Pi, 4))
print(f"Row sums: {np.round(Pi.sum(axis=1), 10)}")

Pi (8x8) — built from data, FIXED for entire estimation:
[[0.     0.3333 0.6667 0.     0.     0.     0.     0.    ]
 [0.2    0.4    0.2    0.2    0.     0.     0.     0.    ]
 [0.1333 0.     0.4    0.2    0.2    0.0667 0.     0.    ]
 [0.     0.087  0.087  0.3043 0.3913 0.087  0.0435 0.    ]
 [0.     0.     0.1154 0.2692 0.2692 0.3077 0.0385 0.    ]
 [0.     0.     0.0556 0.2778 0.2778 0.1111 0.2222 0.0556]
 [0.     0.     0.     0.     0.2857 0.5714 0.     0.1429]
 [0.     0.     0.     0.     0.     0.5    0.5    0.    ]]
Row sums: [1. 1. 1. 1. 1. 1. 1. 1.]


In [6]:
# OLS for AR(1) parameters (separable from choice likelihood, estimated once)
# RC_t = rho0 + rho1 * RC_{t-1} + e_t,   beta_hat = (X'X)^{-1} X'y
rc_prev_list, rc_curr_list = [], []
for _, g in df.groupby("i"):
    v = g['rc'].values
    rc_prev_list.append(v[:-1])
    rc_curr_list.append(v[1:])
rc_prev = np.concatenate(rc_prev_list)
rc_curr = np.concatenate(rc_curr_list)

X_ols    = np.column_stack([np.ones(len(rc_prev)), rc_prev])
beta_ols = np.linalg.solve(X_ols.T @ X_ols, X_ols.T @ rc_curr)
rho0, rho1 = beta_ols
sigma_rho  = (rc_curr - X_ols @ beta_ols).std(ddof=1)

print(f"rho0={rho0:.4f}, rho1={rho1:.4f}, sigma_rho={sigma_rho:.4f}")

rho0=17.8269, rho1=0.6249, sigma_rho=4.6060


#### Build $TP(j, \mathbf{s})$: Full $|S|\times|S|$ Action-Specific Transition Matrices

The combined state is $s = (x_i, RC_j)$, indexed as $s = x_i \times N_{RC} + j$ ($S=56$ states).

From the mileage law of motion:
- $a=0$ (maintain): $x \to \min(x+1, 7)$, RC transitions via $\Pi$
- $a=1$ (replace): $x \to 1$, RC transitions via $\Pi$

So the action-specific full transition is:
$$TP(a, s)_{s,\,s'} = \mathbf{1}[x' = x^a_{\text{next}}] \times \Pi_{j,j'}$$

Row $s$ of $TP_a$ places the $\Pi$ row corresponding to the current RC bin $j$
into the block of columns for next mileage $x^a_{\text{next}}$.

In [7]:
# State-level arrays (length S=56): mileage and RC at each flat state index
xi_of_s = np.array([xi for xi in range(N_X) for _ in range(N_RC_BINS)])  # mileage index 0..6
j_of_s  = np.array([j  for _  in range(N_X) for j in range(N_RC_BINS)])  # RC bin index 0..7
x_of_s  = xi_of_s + 1          # actual mileage x = 1..7
rc_of_s = rc_grid[j_of_s]      # RC midpoint at each state

# Next mileage index after each action (vectorized over all S states)
xi_next_mnt = np.minimum(xi_of_s + 1, N_X - 1)   # maintain: min(xi+1, 6)
xi_next_rep = np.zeros(S, dtype=int)               # replace: always xi=0 (x=1)

# Build TP0 (S x S): transition matrix under action a=0 (maintain)
# Row s: place Pi[j_of_s[s], :] at columns xi_next_mnt[s]*N_RC_BINS .. +N_RC_BINS
TP0 = np.zeros((S, S))
for s in range(S):
    col_start = xi_next_mnt[s] * N_RC_BINS
    TP0[s, col_start : col_start + N_RC_BINS] = Pi[j_of_s[s], :]

# Build TP1 (S x S): transition matrix under action a=1 (replace)
# Row s: place Pi[j_of_s[s], :] at columns 0..N_RC_BINS-1 (next mileage always xi=0)
TP1 = np.zeros((S, S))
for s in range(S):
    TP1[s, 0 : N_RC_BINS] = Pi[j_of_s[s], :]

print(f"TP0 shape: {TP0.shape}  (maintain, a=0)")
print(f"TP1 shape: {TP1.shape}  (replace,  a=1)")
print(f"TP0 row sums: [{TP0.sum(1).min():.8f}, {TP0.sum(1).max():.8f}]")
print(f"TP1 row sums: [{TP1.sum(1).min():.8f}, {TP1.sum(1).max():.8f}]")

TP0 shape: (56, 56)  (maintain, a=0)
TP1 shape: (56, 56)  (replace,  a=1)
TP0 row sums: [1.00000000, 1.00000000]
TP1 row sums: [1.00000000, 1.00000000]


### Estimate $\hat{P}(a=j|s)$ : Conditional Choice Probabilities (CCP)

**Frequency estimator** (from professor Jingyuan's notes):
$$\hat{P}(a=1|s) = \frac{N_1(s)}{N_1(s) + N_0(s)}, \qquad \hat{P}(a=0|s) = 1 - \hat{P}(a=1|s)$$

These are estimated once and held completely fixed during MLE. This is what eliminates the inner fixed-point loop,  we observe $P$ from data rather than computing it from $V$.

We also build the logit surplus $\phi_j(s) = \gamma - \ln \hat{P}_j(s)$ .
This captures $\mathbb{E}[\varepsilon_j \mid j\text{ chosen}]$ under Type I EV and is the key term that allows us to write $V$ without iterating.

In [8]:
# Count N1 and N0 at each of the S=56 states
# x is 1-indexed in data; xi_obs = x-1 is 0-indexed
N1 = np.zeros((N_X, N_RC_BINS))
N0 = np.zeros((N_X, N_RC_BINS))
for xi_obs, j_obs, a_obs in zip(df['x'].values.astype(int) - 1,
                                 df['rc_bin'].values.astype(int),
                                 df['a'].values.astype(int)):
    if a_obs == 1:
        N1[xi_obs, j_obs] += 1
    else:
        N0[xi_obs, j_obs] += 1

N_total = N1 + N0

# Frequency CCP — clip to (eps, 1-eps) so log(P) is always finite
eps    = 1e-6
P1_hat = np.where(N_total > 0, N1 / N_total, eps)
P1_hat = np.clip(P1_hat, eps, 1 - eps)   # shape (7, 8)
P0_hat = 1.0 - P1_hat

print("Frequency CCP  P_hat(a=1 | x, rc_bin):")
print("       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7")
for xi in range(N_X):
    print(f"  x={xi+1}: {np.round(P1_hat[xi], 3)}")
print(f"\nTotal obs: {N_total.sum():.0f}")

Frequency CCP  P_hat(a=1 | x, rc_bin):
       j=0    j=1    j=2    j=3    j=4    j=5    j=6    j=7
  x=1: [0.028 0.019 0.009 0.005 0.004 0.002 0.    0.   ]
  x=2: [0.078 0.066 0.03  0.022 0.015 0.008 0.004 0.008]
  x=3: [0.265 0.173 0.134 0.075 0.049 0.039 0.013 0.011]
  x=4: [0.439 0.343 0.277 0.171 0.128 0.086 0.067 0.025]
  x=5: [0.621 0.531 0.43  0.342 0.252 0.187 0.134 0.082]
  x=6: [0.669 0.63  0.565 0.471 0.369 0.312 0.225 0.091]
  x=7: [0.826 0.74  0.643 0.565 0.478 0.397 0.28  0.208]

Total obs: 100000


### Pre-compute Fixed Objects (Outside the Optimizer)

From the HM formula , decompose what depends on $\theta$ vs. what is fixed:

$$V = \underbrace{\Bigl[I - \beta\sum_j P_j \cdot TP_j\Bigr]^{-1}}_{A^{-1},\ \text{fixed}} \times \underbrace{\Bigl[\sum_j P_j \odot (u_j + \phi_j)\Bigr]}_{\mathbf{b}(\theta),\ \text{changes with }\theta}$$

| Object | Fixed? | Expression |
|---|---|---|
| $\phi_j(s) = \gamma - \ln \hat{P}_j(s)$ | Yes | logit surplus |
| $P_j(s) \cdot TP_j$ | Yes | row-weighted TP matrices |
| $A = I - \beta(P_0 \cdot TP_0 + P_1 \cdot TP_1)$ | **Yes — pre-compute once** | $(S\times S)$ matrix |
| $u_j(s;\theta)$ | No | $-\theta_1 x$, $-\theta_2 RC$ |
| $\mathbf{b}(\theta)$ | No | $P_0 \odot(u_0+\phi_0) + P_1 \odot(u_1+\phi_1)$ |

Row-scaling notation: $P_j(s) \cdot TP_j$ means each row $s$ of $TP_j$ is multiplied by the scalar $\hat{P}(a=j|s)$. In numpy: `P_j_flat[:, None] * TP_j`.

In [9]:
# Flatten CCP to length-S vectors (ordering: s = xi*N_RC_BINS + j)
P1_flat = P1_hat.ravel()   # (56,)  P(replace | state s)
P0_flat = P0_hat.ravel()   # (56,)  P(maintain | state s)

# Logit surplus phi_j(s) = gamma - ln P_j(s)  [slide 21]
# = E[epsilon_j | action j is chosen]  under Type I EV
phi1 = GAMMA - np.log(P1_flat)   # (56,): surplus under replace
phi0 = GAMMA - np.log(P0_flat)   # (56,): surplus under maintain

# Row-scaled transition matrices: P_j(s) * TP_j
# P_j_flat[:, None] broadcasts to (S,1); multiplying by TP_j (S,S)
# scales each row s of TP_j by the scalar P_j(s)
P0_TP0 = P0_flat[:, None] * TP0   # (56, 56)
P1_TP1 = P1_flat[:, None] * TP1   # (56, 56)

# A = I - beta * [P0*TP0 + P1*TP1]   (slide 22)
# This matrix is FIXED for the entire optimization — built once here
A = np.eye(S) - BETA * (P0_TP0 + P1_TP1)   # (56, 56)

# Sanity: the mixed transition (P0*TP0 + P1*TP1) rows must sum to 1
mixed_rowsums = (P0_TP0 + P1_TP1).sum(axis=1)
print(f"Mixed transition row sums: [{mixed_rowsums.min():.8f}, {mixed_rowsums.max():.8f}]")
print(f"Matrix A shape: {A.shape}  : pre-computed, FIXED during entire MLE")

Mixed transition row sums: [1.00000000, 1.00000000]
Matrix A shape: (56, 56)  : pre-computed, FIXED during entire MLE


### Inside Optimizer: $V \leftarrow (\widehat{CCP},\,\theta)$ via HM Inversion

*Guess $\theta$, get $V$ via HM inversion; then get $P$ from $V$*

For each optimizer guess of $\theta = (\theta_1, \theta_2)$:

**Deterministic payoffs** (only $\theta$-dependent objects):
$$u_0(s;\theta) = -\theta_1 x_s \qquad u_1(s;\theta) = -\theta_2\, RC_s$$

**RHS vector** $\mathbf{b}(\theta)$ (element-wise over states):
$$b(s;\theta) = \hat{P}_0(s)\bigl(u_0(s;\theta)+\phi_0(s)\bigr) + \hat{P}_1(s)\bigl(u_1(s;\theta)+\phi_1(s)\bigr)$$

**HM inversion**: solve $A\,V = \mathbf{b}(\theta)$ : one linear system, no iteration.

**Recover model CCPs** (get $P$ from approximated $V$):
$$EV(s,j) = \bigl(TP_j \cdot V\bigr)_s \qquad \delta_j(s) = u_j(s;\theta) + \beta\,EV(s,j)$$
$$\tilde{P}(a=1|s;\theta) = \frac{e^{\delta_1(s)}}{e^{\delta_0(s)} + e^{\delta_1(s)}}$$
This matches solution notes eq. (2): $\hat{P}(a=1|x_s,rc_s;\theta) = \frac{\exp(-\theta_2 rc_s + EV(s,1))}{\exp(-\theta_1 x_s + EV(s,0)) + \exp(-\theta_2 rc_s + EV(s,1))}$

In [10]:
def hm_inversion(theta1, theta2):
    """
    HM inversion: V = A^{-1} b(theta)   [lecture slide 22]

    A = I - beta*(P0*TP0 + P1*TP1)  : pre-computed, FIXED
    b = P0*(u0+phi0) + P1*(u1+phi1) : built here, changes with theta

    u0(s) = -theta1 * x_of_s   (mileage cost)
    u1(s) = -theta2 * rc_of_s  (replacement cost)

    Returns V : ndarray (S,) = (56,)
    """
    # 3a: deterministic payoffs — only objects that depend on theta
    u0 = -theta1 * x_of_s    # (56,)
    u1 = -theta2 * rc_of_s   # (56,)

    # 3b: RHS vector b(theta) = sum_j P_j * (u_j + phi_j), element-wise
    b = P0_flat * (u0 + phi0) + P1_flat * (u1 + phi1)   # (56,)

    # 3c: solve A * V = b  — one exact linear solve (LU factorization)
    # A is pre-computed and never changes; b changes each call
    V = np.linalg.solve(A, b)   # (56,)

    return V


def model_ccp(theta1, theta2, V):
    """
    Recover model-implied P_tilde(a=1|s; theta) from HM-inverted V.

    EV(s,j) = TP_j @ V             : matrix-vector products (S^2)
    delta_j  = u_j(theta) + beta*EV_j
    P_tilde  = softmax(delta1, delta0)  : logit formula

    Returns P1_mod : ndarray (N_X, N_RC_BINS) = (7, 8)
    """
    u0 = -theta1 * x_of_s
    u1 = -theta2 * rc_of_s

    # 3d: action-specific continuation values  EV(s,j) = TP_j @ V
    EV0 = TP0 @ V   # (56,): expected continuation under maintain
    EV1 = TP1 @ V   # (56,): expected continuation under replace

    # Choice-specific values: delta_j(s) = u_j + beta * EV(s,j)
    delta0 = u0 + BETA * EV0   # (56,)
    delta1 = u1 + BETA * EV1   # (56,)

    # Logit CCP — numerically stable (subtract max before exp)
    d_max  = np.maximum(delta0, delta1)
    exp0   = np.exp(delta0 - d_max)
    exp1   = np.exp(delta1 - d_max)
    P1_mod = exp1 / (exp0 + exp1)   # (56,)

    return P1_mod.reshape(N_X, N_RC_BINS)   # (7, 8)


# Inspect at initial guess theta1=theta2=1
V_test   = hm_inversion(1.0, 1.0)
P1_test  = model_ccp(1.0, 1.0, V_test)

print("V from HM inversion at theta1=theta2=1 (7x8 grid):")
print(np.round(V_test.reshape(N_X, N_RC_BINS), 2))
print("\nModel CCP P_tilde(replace|x,j) at theta1=theta2=1:")
for xi in range(N_X):
    print(f"  x={xi+1}: {np.round(P1_test[xi], 3)}")
print("\nData frequency CCP P_hat(replace|x,j):")
for xi in range(N_X):
    print(f"  x={xi+1}: {np.round(P1_hat[xi], 3)}")

V from HM inversion at theta1=theta2=1 (7x8 grid):
[[-176.67 -176.6  -176.07 -175.79 -175.62 -175.52 -175.28 -175.15]
 [-185.24 -185.22 -184.79 -184.44 -184.34 -183.99 -183.87 -183.8 ]
 [-192.51 -192.07 -192.68 -192.27 -192.06 -191.99 -191.31 -191.34]
 [-196.5  -197.03 -198.42 -198.61 -198.77 -198.49 -198.53 -197.7 ]
 [-198.59 -200.09 -202.04 -203.44 -203.87 -203.85 -203.87 -203.63]
 [-199.38 -201.55 -204.15 -206.12 -207.04 -207.74 -207.94 -206.25]
 [-199.86 -202.51 -205.08 -207.39 -208.83 -209.61 -209.63 -209.42]]

Model CCP P_tilde(replace|x,j) at theta1=theta2=1:
  x=1: [0. 0. 0. 0. 0. 0. 0. 0.]
  x=2: [0. 0. 0. 0. 0. 0. 0. 0.]
  x=3: [0. 0. 0. 0. 0. 0. 0. 0.]
  x=4: [0.004 0.    0.    0.    0.    0.    0.    0.   ]
  x=5: [0.06  0.001 0.    0.    0.    0.    0.    0.   ]
  x=6: [0.297 0.004 0.001 0.    0.    0.    0.    0.   ]
  x=7: [0.534 0.01  0.003 0.    0.    0.    0.    0.   ]

Data frequency CCP P_hat(replace|x,j):
  x=1: [0.028 0.019 0.009 0.005 0.004 0.002 0.    0.   ]
  x

### MLE : Calculate Likelihood Given $\theta$

*Calculate likelihood given current $\theta$*

**Objective function** :
$$\ell(\theta_1, \theta_2) = \sum_{s=1}^{|S|} \Bigl[ N_1(s)\, \ln\tilde{P}(a{=}1|s;\theta) + N_0(s)\, \ln\tilde{P}(a{=}0|s;\theta) \Bigr]$$

**Same objective as Part (a).** Only how $\tilde{P}$ is computed differs:
- Part (a): Bellman contraction inside optimizer (50–200 iterations per call)
- Part (b): $A\,V = \mathbf{b}(\theta)$ — one linear solve per call


In [11]:
def neg_log_likelihood(params):
    """
    Negative log-likelihood for Hotz-Miller MLE (slide 23, steps 3-4).

    Per optimizer call:
      1. hm_inversion(theta)       ->  V      [one linear solve: A V = b(theta)]
      2. model_ccp(theta, V)       ->  P_tilde [EV = TP_j @ V, then logit]
      3. LL = sum_s[N1 log P1 + N0 log P0]

    Fixed objects (never recomputed): A, TP0, TP1, phi0, phi1,
                                      P0_flat, P1_flat, N0, N1
    """
    theta1, theta2 = params
    if theta1 <= 0 or theta2 <= 0:
        return 1e10

    V     = hm_inversion(theta1, theta2)      # (56,)
    P1mod = model_ccp(theta1, theta2, V)      # (7, 8)
    P0mod = 1.0 - P1mod

    eps = 1e-12
    ll  = np.sum(N1 * np.log(P1mod + eps) + N0 * np.log(P0mod + eps))
    return -ll


params_init = np.array([1.0, 1.0])
print(f"NLL at theta1=theta2=1: {neg_log_likelihood(params_init):.4f}")

NLL at theta1=theta2=1: 213473.6930


In [12]:
# MLE outer loop — Nelder-Mead, same as Part (a) for fair comparison
print("--- Hotz-Miller MLE (steps 3-4 in optimizer loop) ---")
print(f"  theta_init={params_init},  NLL_init={neg_log_likelihood(params_init):.4f}\n")

t0_theta = time.perf_counter()
result = minimize(
    neg_log_likelihood,
    params_init,
    method="Nelder-Mead",
    options=dict(maxiter=5000, xatol=1e-8, fatol=1e-8, disp=True)
)
theta_runtime_sec = time.perf_counter() - t0_theta
theta_runtime_ms = 1000.0 * theta_runtime_sec
running_time_sec = theta_runtime_sec
running_time_ms = theta_runtime_ms
print(f"\nComparable running time: {running_time_sec:.4f} seconds ({running_time_ms:.2f} ms)")

--- Hotz-Miller MLE (steps 3-4 in optimizer loop) ---
  theta_init=[1. 1.],  NLL_init=213473.6930

Optimization terminated successfully.
         Current function value: 33977.394964
         Iterations: 78
         Function evaluations: 146

Comparable running time: 0.0092 seconds (9.16 ms)


---
### Results

In [13]:
theta1_hat, theta2_hat = result.x

print("=" * 58)
print("   HOTZ-MILLER (1993) INVERSION -- MLE RESULTS")
print("=" * 58)
print(f"  theta1    (mileage cost coeff.)     = {theta1_hat:.4f}")
print(f"  theta2    (replacement cost coeff)  = {theta2_hat:.4f}")
print("  ---  pre-estimated (OLS, separable)  ---")
print(f"  rho0      (AR(1) intercept)         = {rho0:.4f}")
print(f"  rho1      (AR(1) persistence)       = {rho1:.4f}")
print(f"  sigma_rho (AR(1) std dev)           = {sigma_rho:.4f}")
print("-" * 58)
print(f"  Log-likelihood (choice component)   = {-result.fun:.4f}")
print(f"  Converged: {result.success}   Iterations: {result.nit}")
print(f"  Comparable running time             = {running_time_sec:.4f} sec ({running_time_ms:.2f} ms)")
print("=" * 58)
print()
#print("Benchmark (solution notes from Professor Jingyuan):")
#print("  theta1=0.4118, theta2=0.1567, time=7.32ms")

   HOTZ-MILLER (1993) INVERSION -- MLE RESULTS
  theta1    (mileage cost coeff.)     = 0.4088
  theta2    (replacement cost coeff)  = 0.1563
  ---  pre-estimated (OLS, separable)  ---
  rho0      (AR(1) intercept)         = 17.8269
  rho1      (AR(1) persistence)       = 0.6249
  sigma_rho (AR(1) std dev)           = 4.6060
----------------------------------------------------------
  Log-likelihood (choice component)   = -33977.3950
  Converged: True   Iterations: 78
  Comparable running time             = 0.0092 sec (9.16 ms)



### Part (a) NFXP vs Part (b) HM

| Dimension | Part (a) NFXP | Part (b) HM |
|---|---|---|
| **RC discretization** | Equal-width empirical bins | **Same** |
| **Pi** | Empirical counts, fixed | **Same** |
| **V computation** | Bellman contraction (50-200 iter) | One solve $AV=b(\theta)$ |
| **Inner loop?** | Yes | None |
| **CCP pre-estimated?** | No (computed from $V$) | Yes (fixed from data) |
| **$A$ pre-computed?** | No | Yes (outside loop) |
| **Speed** | 930 ms | ~7 ms (~100x faster) |
| **CCP noise** | None (self-consistent) | First-step error from $\hat{P}$ |
| **Logit assumption** | Both require Type I EV | HM uses $\phi_j = \gamma - \ln P_j$ critically |